# Halo Mass Function Plotting
This notebook provides helpers to load and plot halo mass functions.

In [ ]:
import sys
from pathlib import Path

HICOLA_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'hicola_paths.py').exists())
sys.path.insert(0, str(HICOLA_ROOT / 'scripts'))
import hicola_paths as hp

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def load_mass_function(file_path, skiprows=0, delimiter=None, comment='#'):
    """Load a halo mass function from a text file.
    Expects at least two columns: mass and dn/dlogM.
    Returns (mass, dn/dlogM).
    """
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f'Mass function file not found: {path}')
    data = np.loadtxt(path, delimiter=delimiter, comments=comment, skiprows=skiprows)
    if data.ndim != 2 or data.shape[1] < 2:
        raise ValueError(f'Expected at least two columns in {path}, got shape {data.shape}')
    return data[:, 0], data[:, 1]

def plot_mass_function(file_paths, labels=None, colors=None, ax=None, *,
                       xscale='log', yscale='log',
                       xlabel=r'$M_\mathrm{halo}\, [M_\odot/h]$',
                       ylabel=r'$\mathrm{d}n/\mathrm{d}\log M\, [h^3\,\mathrm{Mpc}^{-3}]$',
                       title='Halo Mass Function',
                       figsize=(8, 6),
                       linewidth=1.8,
                       marker='o',
                       markersize=4,
                       grid=True,
                       legend=True,
                       legend_loc='best',
                       style=None,
                       y='dndlogM'):
    """Plot dn/dlogM or cumulative n(>M).
    y: 'dndlogM' (default) or 'n_gt_M' (cumulative).
    """
    if isinstance(file_paths, (str, Path)):
        file_paths = [file_paths]
    file_paths = [Path(f) for f in file_paths]
    if labels is None:
        labels = [p.stem for p in file_paths]
    if len(labels) != len(file_paths):
        raise ValueError('labels must match number of file paths')
    if colors is None:
        colors = [None] * len(file_paths)

    if style is not None:
        plt.style.use(style)
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    for path, label, color in zip(file_paths, labels, colors):
        mass, phi = load_mass_function(path)  # phi = dn/dlogM

        if y == 'dndlogM':
            yvals = phi
            ylab = ylabel
        elif y == 'n_gt_M':
            logm = np.log10(mass)
            dlog = np.empty_like(logm)
            if len(logm) > 1:
                dlog[1:-1] = 0.5 * (logm[2:] - logm[:-2])
                dlog[0] = logm[1] - logm[0]
                dlog[-1] = logm[-1] - logm[-2]
            else:
                dlog[0] = 0.0
            dn = phi * dlog
            n_gt = np.cumsum(dn[::-1])[::-1]
            yvals = n_gt
            ylab = r'$n(>M)\, [h^3\,\mathrm{Mpc}^{-3}]$'
        else:
            raise ValueError("Unknown y option; choose 'dndlogM' or 'n_gt_M'")

        ax.plot(mass, yvals,
                label=label,
                color=color,
                linewidth=linewidth,
                marker=marker,
                markersize=markersize,
                markevery=max(len(mass) // 24, 1))

    ax.set_xscale(xscale)
    ax.set_yscale(yscale)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylab, fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='semibold')
    ax.grid(grid, which='both', linestyle='--', linewidth=0.5, alpha=0.65)
    if legend:
        ax.legend(loc=legend_loc, fontsize=10)
    fig.tight_layout()
    return fig, ax

In [ ]:
from pathlib import Path

file_paths = [
    hp.standalone_run_dir('LCDM_reduced') / 'snapshot_LCDM_reduced_z0.000' / 'massfunc_z0.000.txt',
    hp.standalone_run_dir('cuGal_reduced') / 'snapshot_cuGal_reduced_z0.000' / 'massfunc_z0.000.txt',
    hp.standalone_run_dir('nDGP_reduced') / 'snapshot_nDGP_reduced_z0.000' / 'massfunc_z0.000.txt'
]
labels = ['LCDM_reduced', 'cuGal_reduced', 'nDGP_reduced']

# Example: plot cumulative number density n(>M)
fig, ax = plot_mass_function(file_paths, labels=labels, y='n_gt_M',
                              title='Halo Cumulative Number Density Comparison at z=0',
                              xlabel=r'$M_\mathrm{halo}\, [M_\odot/h]$')
ax.set_xlim(1e10, 1e16)
ax.set_ylim(1e-12, 1e-1)


In [ ]:
# Robust HMF comparison: handles missing snapshot files, sorts before interpolation, and avoids divide-by-zero
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


def _sort_mass_phi(m, phi):
    m = np.array(m)
    phi = np.array(phi)
    idx = np.argsort(m)
    return m[idx], phi[idx]


def _safe_load(path):
    try:
        return load_mass_function(path)
    except Exception as e:
        print(f'Could not load {path}:', e)
        return None, None


def _safe_ratio(numer, denom):
    numer = np.asarray(numer, dtype=float)
    denom = np.asarray(denom, dtype=float)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.empty_like(numer, dtype=float)
        mask = denom > 0
        ratio[~mask] = np.nan
        ratio[mask] = numer[mask] / denom[mask]
    return ratio

pairs = [(0,'1.650'), (1,'1.400'), (2,'1.200'), (3,'1.000')]
for x, z in pairs:
    # Comparison (Bart) HMFs, Box1
    m_l, phi_l = _safe_load(hp.reference_file('COLA_NM4096', 'LCDM', 1, float(z), 'HMF'))
    m_n, phi_n = _safe_load(hp.reference_file('COLA_NM4096', 'nDGP_N1.2', 1, float(z), 'HMF'))
    if m_l is None or m_n is None:
        print(f'Skipping x={x} (missing Bart files)')
        continue
    m_l, phi_l = _sort_mass_phi(m_l, phi_l)
    m_n, phi_n = _sort_mass_phi(m_n, phi_n)

    # Candidate snapshot mass function paths (try multiple fallbacks)
    cand_L = [
        hp.snapshot_dir('COLA_NM4096', 'LCDM', 's1234', float(z)) / f'massfunc_z{z}.txt',
        hp.standalone_run_dir('LCDM_reduced') / f'snapshot_LCDM_reduced_z{z}' / f'massfunc_z{z}.txt',
        hp.standalone_run_dir('LCDM') / f'snapshot_LCDM_z{z}' / f'massfunc_z{z}.txt',
    ]
    cand_N = [
        hp.snapshot_dir('COLA_NM4096', 'nDGP_N1.2', 's1234', float(z)) / f'massfunc_z{z}.txt',
        hp.standalone_run_dir('nDGP_reduced') / f'snapshot_nDGP_reduced_z{z}' / f'massfunc_z{z}.txt',
        hp.standalone_run_dir('nDGP') / f'snapshot_nDGP_z{z}' / f'massfunc_z{z}.txt',
    ]

    mpl = phipl = mpn = phipn = None
    for p in cand_L:
        mpl, phipl = _safe_load(p)
        if mpl is not None:
            mpl, phipl = _sort_mass_phi(mpl, phipl)
            break
    for p in cand_N:
        mpn, phipn = _safe_load(p)
        if mpn is not None:
            mpn, phipn = _sort_mass_phi(mpn, phipn)
            break

    # Bart ratio (interpolated onto Bart LCDM mass grid) - use safe division
    try:
        phi_n_on_l = np.interp(np.log10(m_l), np.log10(m_n), phi_n)
        ratio_bart = _safe_ratio(phi_n_on_l, phi_l)
    except Exception as e:
        print(f'Could not compute Bart ratio for x={x}, z={z}:', e)
        continue

    fig, ax = plt.subplots(figsize=(7,5))
    ax.plot(m_l, ratio_bart, label='Bart ratio (nDGP/LCDM)', lw=2)

    # If snapshot massfuncs found, compute and plot their ratio (interpolating numerator if needed)
    if mpl is not None and mpn is not None and len(phipl) and len(phipn):
        try:
            # interpolate nDGP snapshot onto LCDM snapshot mass grid before safe division
            phipn_on_pl = np.interp(np.log10(mpl), np.log10(mpn), phipn)
            ratio_snap = _safe_ratio(phipn_on_pl, phipl)
            ax.plot(mpl, ratio_snap, label='snapshot ratio (nDGP/LCDM)', lw=2, ls='--')
        except Exception as e:
            print(f'Could not compute snapshot ratio for z={z}:', e)
    else:
        print(f'No snapshot mass function pair found for z={z}; plotted only Bart ratio.')

    ax.set_xscale('log')
    ax.set_xlabel(r'$M_\mathrm{halo}\, [M_\odot/h]$')
    ax.set_ylabel('phi_n/phi_l')
    ax.set_title(f'HMF ratio comparison at z={z}')
    ax.legend()
    ax.grid(True, which='both', ls=':', alpha=0.6)
    plt.tight_layout()
    plt.show()
